# 03 — The traditional baseline

Before claiming an LLM method is worth hours of local inference, you have to say
what it is worth *more than*. This notebook builds that comparison: five
ordinary scikit-learn classifiers over ordinary text features, no LLM anywhere,
half a minute on Movie and a couple of minutes on VCBench.

The honest answer, stated up front so the rest of the notebook can support it
rather than sell it: **on VCBench the traditional suite beats every reasoning
method and their ensemble.** On Movie it is the other way round, and by a
thinner margin. Traditional ML is a real baseline here, not a strawman set up
to lose.

Everything that carries a number reads from `precomputed/`. One optional cell
at the end refits the suite live from the public HuggingFace copy of Movie, and
skips itself without a network.

In [1]:
import importlib.util
from pathlib import Path

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score


def find_example_root(start: Path | None = None) -> Path:
    """Walk up from `start` (default: the CWD) to the example dir holding precomputed/."""
    start = (start or Path.cwd()).resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "examples" / "vcbench-movie-local"):
            if (candidate / "precomputed").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not find the example directory (the one containing precomputed/) "
        f"above {start}. Run this notebook from inside the example."
    )


EXAMPLE_ROOT = find_example_root()
PRECOMPUTED = EXAMPLE_ROOT / "precomputed"

# `id` is the founder_uuid on VCBench and the IMDb tconst on Movie.
SPLITS = {"VCBench public": "vcbench_public", "Movie test": "movie_test"}
METHODS = {"PI": "pi", "RRF": "rrf", "GPTree": "gptree", "RRM": "rrm"}


def score_ranking(labels, scores) -> dict[str, float]:
    return {
        "ROC-AUC": roc_auc_score(labels, scores),
        "PR-AUC": average_precision_score(labels, scores),
    }


def load_scores(prefix: str, series: str) -> pd.DataFrame:
    return pd.read_csv(PRECOMPUTED / f"{prefix}_{series}_scores.csv").set_index("id")


# The traditional file is the one that carries the label on both splits: the
# reasoning files for a held-out split ship scores only.
labels = {name: load_scores(prefix, "traditional")["label"] for name, prefix in SPLITS.items()}

for name, y in labels.items():
    print(f"{name}: n={len(y):,}, base rate {y.mean():.3f}")

VCBench public: n=4,500, base rate 0.090
Movie test: n=727, base rate 0.208


## What the suite is

One pipeline, run identically on both datasets. Nothing here is tuned per
dataset, which is the point: it is the thing you would try first.

**Features**, all derived from the same free text the LLM methods read
(`anonymised_prose` on VCBench, the plot summary on Movie):

| Block | Width | What it is |
|---|---:|---|
| Text length | 1 | `len(text)`, the cheapest signal there is |
| Sentence embedding | 384 | `all-MiniLM-L6-v2`, normalised |
| TF-IDF | 400 | top 400 English terms |

**Models**, all with `class_weight="balanced"` where they support it, seed 42:

- Logistic regression (`max_iter=1500`, on scaled features)
- HistGradientBoosting
- RandomForest, 250 trees
- ExtraTrees, 250 trees
- GaussianNB (on scaled features)

**Discipline.** Every transformer is fitted on train and only applied to test:
the TF-IDF vocabulary, the median imputer, the scaler. Fitting the vectoriser
on the full dataset is the single most common way a text baseline quietly
inflates itself, and it is worth checking for in any baseline you are handed.

**Ensemble.** Convert each model's probabilities to percentile ranks, average
the five. Same operator as the reasoning ensemble in notebook 01, for the same
reason: the five models are calibrated differently and averaging raw
probabilities lets the most confident one win by default.

The code is `scripts/run_traditional.py`. It is about 300 lines and worth
reading before you trust any of the numbers below.

In [2]:
traditional = {
    name: score_ranking(labels[name], load_scores(prefix, "traditional")["score"])
    for name, prefix in SPLITS.items()
}

trad_table = pd.DataFrame(traditional).T
trad_table.index.name = "Split"

REFERENCE_TRADITIONAL = {"VCBench public": (0.7382, 0.2258), "Movie test": (0.6354, 0.2927)}
for name, (roc, pr) in REFERENCE_TRADITIONAL.items():
    row = trad_table.loc[name]
    assert abs(row["ROC-AUC"] - roc) < 1e-3, (name, "ROC-AUC", row["ROC-AUC"], roc)
    assert abs(row["PR-AUC"] - pr) < 1e-3, (name, "PR-AUC", row["PR-AUC"], pr)

print("Five-model rank-average ensemble, matching the reference run to within 0.001.\n")
trad_table.round(4)

Five-model rank-average ensemble, matching the reference run to within 0.001.



,ROC-AUC,PR-AUC
Split,,
VCBench public,0.7382,0.2258
Movie test,0.6354,0.2927


## Reproducing the score files

The two CSVs you just read were written by `scripts/run_traditional.py`, which
takes any records file with a text column and a label column.

**VCBench** is not publicly downloadable. Request it at
[vcbench.com](https://vcbench.com), then run the same 3-fold out-of-fold
protocol the shipped file uses:

```bash
export VCBENCH_DATA=~/.trl-data/vcbench/vcbench_final_public.csv

python scripts/run_traditional.py \
  --train "$VCBENCH_DATA" --kfold 3 \
  --text-fields anonymised_prose --label-col success \
  --out-dir results/vcbench_traditional
```

Out-of-fold rather than a single holdout because the public split is all you
have: 4,500 founders at a 9% base rate is 405 positives, and a 20% holdout
would put 81 of them in the test set. Three folds spend every founder once as
test data and keep the estimate steady.

Nothing in this repo will fetch VCBench for you, and no raw VCBench text is
committed here.

**Movie** is a public HuggingFace dataset, so the next cell reruns the whole
thing from scratch.

## Optional: refit the suite on Movie, live

This cell downloads `Francis2003/Movie-O-Label`, rebuilds the temporal split
(train before 2013, test from 2013 on), and runs the same pipeline the shipped
scores came from. About half a minute.

It needs the `[examples]` extras and a network. Without them it prints a note
and moves on, so the notebook still runs end to end.

Two details the raw dataset makes you deal with, both of which the shipped
scores already handle:

- The three HuggingFace splits are a random 60/20/20 and are not the split this
  example uses. Concatenate them and re-split on `year`, so the model is never
  trained on films made after the ones it is tested on.
- 2,200 rows hold 2,188 distinct `imdb_id`s. Twelve films appear twice with the
  same label. Deduplicate before scoring or those twelve count double.

Expect the ensemble to land near the shipped 0.6354, not exactly on it: the two
tree ensembles in the suite are sensitive to the scikit-learn version they were
fitted under, and the shipped file came from the reference run's version. The
asserted number in this notebook is the precomputed one.

In [ ]:
MOVIE_DATASET = "Francis2003/Movie-O-Label"
SPLIT_YEAR = 2013

try:
    from datasets import load_dataset

    # Import the runner's pipeline rather than restating it, so this cell and
    # the shipped scores go through exactly the same code.
    spec = importlib.util.spec_from_file_location(
        "run_traditional", EXAMPLE_ROOT / "scripts" / "run_traditional.py"
    )
    run_traditional = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(run_traditional)

    raw = load_dataset(MOVIE_DATASET)
    films = (
        pd.concat([raw[split].to_pandas() for split in raw], ignore_index=True)
        .drop_duplicates(subset="imdb_id", keep="first")
        .reset_index(drop=True)
    )
    train = films[films["year"] < SPLIT_YEAR].reset_index(drop=True)
    test = films[films["year"] >= SPLIT_YEAR].reset_index(drop=True)
    print(f"{len(films):,} distinct films · train {len(train):,} (base {train['nominated'].mean():.3f}) "
          f"· test {len(test):,} (base {test['nominated'].mean():.3f})\n")

    X_train, X_test, feature_info = run_traditional.build_features(
        train, test, ["summary"], [], "sentence-transformers/all-MiniLM-L6-v2", 400
    )
    print(f"feature dim {feature_info['n_features_total']} "
          f"(1 length + {feature_info['n_features_embedding']} embedding "
          f"+ {feature_info['n_features_tfidf']} tf-idf)\n")

    results, predictions = run_traditional.run_train_test(
        X_train, train["nominated"].to_numpy(), X_test, test["nominated"].to_numpy(), 42
    )

    shipped = load_scores("movie_test", "traditional")["score"]
    live = pd.Series(predictions["ensemble"].to_numpy(), index=test["imdb_id"]).reindex(shipped.index)
    print(f"\nshipped ensemble : ROC-AUC {traditional['Movie test']['ROC-AUC']:.4f} · "
          f"PR-AUC {traditional['Movie test']['PR-AUC']:.4f}")
    print(f"this rerun       : ROC-AUC {results['ensemble']['roc_auc']:.4f} · "
          f"PR-AUC {results['ensemble']['pr_auc']:.4f}")
    print(f"rank correlation between the two rankings: {live.corr(shipped, method='spearman'):.3f}")
except Exception as exc:
    print(f"Skipping the live refit: {type(exc).__name__}: {exc}")
    print(f"It needs the `datasets` and `sentence-transformers` extras and network access "
          f"to HuggingFace. Every number asserted in this notebook comes from precomputed/ "
          f"and does not.")

## Where the baseline actually lands

Both families, both datasets, all four numbers computed from files in
`precomputed/`.

In [4]:
def reasoning_ensemble(prefix: str, index: pd.Index) -> pd.Series:
    """Mean percentile rank of the four reasoning methods, as in notebook 01."""
    wide = pd.DataFrame(
        {name: load_scores(prefix, key)["score"] for name, key in METHODS.items()}
    ).reindex(index)
    return wide.rank(pct=True).mean(axis=1)


comparison = pd.DataFrame(
    {
        name: {
            ("Traditional", metric): value for metric, value in traditional[name].items()
        }
        | {
            ("Reasoning", metric): value
            for metric, value in score_ranking(
                labels[name], reasoning_ensemble(prefix, labels[name].index)
            ).items()
        }
        for name, prefix in SPLITS.items()
    }
).T
comparison.index.name = "Split"

REFERENCE_REASONING = {"VCBench public": (0.7188, 0.2072), "Movie test": (0.6515, 0.3007)}
for name, (roc, pr) in REFERENCE_REASONING.items():
    assert abs(comparison.loc[name, ("Reasoning", "ROC-AUC")] - roc) < 1e-3, (name, "ROC-AUC")
    assert abs(comparison.loc[name, ("Reasoning", "PR-AUC")] - pr) < 1e-3, (name, "PR-AUC")

comparison.round(4)

Traditional         Reasoning        
                   ROC-AUC  PR-AUC   ROC-AUC  PR-AUC
Split                                               
VCBench public      0.7382  0.2258    0.7188  0.2072
Movie test          0.6354  0.2927    0.6515  0.3007

So the ordering flips with the dataset:

- **VCBench.** Traditional 0.7382 / 0.2258 against reasoning 0.7188 / 0.2072.
  Five sklearn models over MiniLM embeddings beat four LLM methods and their
  ensemble, on both metrics, in a couple of minutes against four LLM passes
  that each take hours.
- **Movie.** Reasoning 0.6515 / 0.3007 against traditional 0.6354 / 0.2927. The
  reasoning side wins, by 0.016 ROC-AUC on 727 films, which is inside sampling
  noise. Treat it as "competitive", not "better".

Neither result is the pitch you usually hear for LLM-based ML, and both are what
the shipped scores say. Two things follow.

**Anyone comparing methods without this baseline is not telling you much.** A
reasoning method at 0.72 ROC-AUC sounds strong until a two-minute baseline
returns 0.74 on the same split.

**"Which family wins" is the less interesting question.** The two families are
reading the same text through very different lenses, so the useful question is
whether their mistakes are different enough to be worth combining. On both
datasets they are, and that is notebook 04.

## Where next

- **[04 — Ensembling](04_ensembling.ipynb)** combines the two families and beats
  both of them, on both metrics, on both datasets.
- **[02 — Reasoning methods](02_reasoning_methods.ipynb)** if you skipped it:
  the artifacts these baselines are being compared against are readable text,
  which no number in this notebook captures.